<a href="https://colab.research.google.com/github/rahu2004/postgress_SQL/blob/main/Global_HR_%26_Sales_Analytics_System_using_PostgressSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Start PostgreSQL

In [50]:
!apt update -y
!apt install postgresql postgresql-contrib -y
!service postgresql start

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,818 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
17 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pa

Setup DB User + Database

In [51]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'root';"
!sudo -u postgres createdb analytics_db

ALTER ROLE
createdb: error: database creation failed: ERROR:  database "analytics_db" already exists


Connect from Python

In [52]:
import psycopg2

conn = psycopg2.connect(
    dbname="analytics_db",
    user="postgres",
    password="root",
    host="localhost"
)

cur = conn.cursor()
print("Connected successfully")

Connected successfully


CLEAN RESET

In [53]:
conn.rollback()

cur.execute("""
DROP TABLE IF EXISTS sales CASCADE;
DROP TABLE IF EXISTS employees CASCADE;
DROP TABLE IF EXISTS products CASCADE;
DROP TABLE IF EXISTS departments CASCADE;
""")

conn.commit()
print("Clean slate ready")

Clean slate ready


create tables

In [54]:
cur.execute("""
CREATE TABLE departments (
    dept_id SERIAL PRIMARY KEY,
    dept_name VARCHAR(50)
);

CREATE TABLE employees (
    emp_id SERIAL PRIMARY KEY,
    emp_name VARCHAR(100),
    manager_id INT,
    dept_id INT REFERENCES departments(dept_id),
    salary INT
);

CREATE TABLE products (
    product_id SERIAL PRIMARY KEY,
    product_name VARCHAR(100),
    category VARCHAR(50),
    price NUMERIC
);

CREATE TABLE sales (
    sale_id SERIAL PRIMARY KEY,
    emp_id INT REFERENCES employees(emp_id),
    product_id INT REFERENCES products(product_id),
    quantity INT,
    sale_date DATE
);
""")

conn.commit()
print("Tables created")

Tables created


insert into tables

dept

In [55]:
cur.execute("""
INSERT INTO departments (dept_name) VALUES
('Engineering'),
('Sales'),
('Marketing');
""")
conn.commit()

Employees

In [56]:
cur.execute("""
INSERT INTO employees (emp_name, manager_id, dept_id, salary) VALUES
('Alice', NULL, 1, 100000),
('Bob', 1, 1, 80000),
('Charlie', 1, 2, 75000),
('David', 3, 2, 70000);
""")
conn.commit()

products

In [57]:
cur.execute("""
INSERT INTO products (product_name, category, price) VALUES
('Laptop', 'Tech', 1000),
('Phone', 'Tech', 700),
('CRM Tool', 'Software', 500);
""")
conn.commit()

sales

In [58]:
cur.execute("""
INSERT INTO sales (emp_id, product_id, quantity, sale_date) VALUES
(2, 1, 2, '2025-01-10'),
(2, 2, 3, '2025-01-11'),
(3, 3, 1, '2025-01-12'),
(4, 1, 1, '2025-01-13');
""")
conn.commit()

VERIFY DATA

In [59]:
cur.execute("SELECT * FROM departments;")
print("Departments:", cur.fetchall())

cur.execute("SELECT * FROM employees;")
print("Employees:", cur.fetchall())

cur.execute("SELECT * FROM products;")
print("Products:", cur.fetchall())

cur.execute("SELECT * FROM sales;")
print("Sales:", cur.fetchall())

Departments: [(1, 'Engineering'), (2, 'Sales'), (3, 'Marketing')]
Employees: [(1, 'Alice', None, 1, 100000), (2, 'Bob', 1, 1, 80000), (3, 'Charlie', 1, 2, 75000), (4, 'David', 3, 2, 70000)]
Products: [(1, 'Laptop', 'Tech', Decimal('1000')), (2, 'Phone', 'Tech', Decimal('700')), (3, 'CRM Tool', 'Software', Decimal('500'))]
Sales: [(1, 2, 1, 2, datetime.date(2025, 1, 10)), (2, 2, 2, 3, datetime.date(2025, 1, 11)), (3, 3, 3, 1, datetime.date(2025, 1, 12)), (4, 4, 1, 1, datetime.date(2025, 1, 13))]


Ranking Sales Performance (WINDOW FUNCTION)

In [60]:
cur.execute("""
SELECT
    e.emp_name,
    SUM(s.quantity * p.price) AS total_sales,
    RANK() OVER (ORDER BY SUM(s.quantity * p.price) DESC) AS rank
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
JOIN products p ON s.product_id = p.product_id
GROUP BY e.emp_name;
""")

print(cur.fetchall())

[('Bob', Decimal('4100'), 1), ('David', Decimal('1000'), 2), ('Charlie', Decimal('500'), 3)]


Advanced KPI View

In [62]:
cur.execute("""
CREATE VIEW sales_kpi AS
SELECT
    e.emp_name,
    d.dept_name,
    SUM(s.quantity * p.price) AS revenue
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
JOIN departments d ON e.dept_id = d.dept_id
JOIN products p ON s.product_id = p.product_id
GROUP BY e.emp_name, d.dept_name;
""")

conn.commit()

Stored Procedure

In [63]:
cur.execute("""
CREATE OR REPLACE FUNCTION get_employee_sales(emp INT)
RETURNS INT AS $$
DECLARE total INT;
BEGIN
    SELECT SUM(s.quantity * p.price)
    INTO total
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    WHERE s.emp_id = emp;

    RETURN COALESCE(total, 0);
END;
$$ LANGUAGE plpgsql;
""")

conn.commit()

call procedure

In [64]:
cur.execute("SELECT get_employee_sales(2);")
print(cur.fetchone())

(4100,)


Business Insight Queries

In [65]:
cur.execute("""
SELECT p.category, SUM(s.quantity) as total_units
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category;
""")
print(cur.fetchall())

[('Tech', 6), ('Software', 1)]


Save SQL Files

In [66]:
schema_sql = """
CREATE TABLE departments (
    dept_id SERIAL PRIMARY KEY,
    dept_name VARCHAR(50)
);

CREATE TABLE employees (
    emp_id SERIAL PRIMARY KEY,
    emp_name VARCHAR(100),
    manager_id INT,
    dept_id INT REFERENCES departments(dept_id),
    salary INT
);
"""

with open("schema.sql", "w") as f:
    f.write(schema_sql)

print("schema.sql created")

schema.sql created


In [67]:
queries_sql = """
SELECT * FROM employees;

SELECT
    emp_name,
    salary,
    RANK() OVER (ORDER BY salary DESC)
FROM employees;
"""

with open("queries.sql", "w") as f:
    f.write(queries_sql)

print("queries.sql created")

queries.sql created


Download files

In [68]:
from google.colab import files

files.download("schema.sql")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
files.download("queries.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>